# MPECSS - NosBench Benchmark (Group 6 of 6)This notebook runs **Group 6** of the NosBench benchmark suite.- **Dataset**: NosBench (Group 6)- **Problems**: 100 (balanced small/medium/large split)- **Resume support**: built in via `kaggle_setup/resumable_benchmark.py`**Prerequisites:**1. Add a Kaggle Input dataset that contains `benchmarks/nosbench/nosbench-json`2. Enable Internet accessRun the cells in order. After a Kaggle restart, re-run cells 1-3, then use the **Resume** cell.

## 1. Configuration

In [ ]:
# NosBench Group 6 Configuration
import os
from pathlib import Path

DATASET = 'nosbench'
GROUP = 6
RUN_TAG = f'Kaggle_NosBench_Group{GROUP}'

WORKERS = 4
TIMEOUT = 3600
NUM_PROBLEM = None  # None = all problems; set 5/10/etc for quick tests
SAVE_LOGS = True
SHUFFLE = True

# Repository (code)
REPO_DIR = "/kaggle/working/mpecss-kaggle"
REPO_GIT_URL = "https://github.com/mpecssalgorithm/mpecss.git"

# IMPORTANT: Results go directly to /kaggle/working/outputs to persist after session ends
OUTPUT_DIR = "/kaggle/working/outputs"

DATASET_JSON_DIR = 'nosbench-json'
EXPECTED_PROBLEM_COUNT = 603
KAGGLE_INPUT_ROOT = Path('/kaggle/input')


def _count_json_files(path):
    path = Path(path)
    if not path.is_dir():
        return 0
    return sum(1 for item in path.iterdir() if item.is_file() and item.name.endswith('.json'))


def _resolve_dataset_path():
    checked = []
    seen = set()
    candidates = [Path(REPO_DIR) / 'benchmarks' / DATASET / DATASET_JSON_DIR]

    if KAGGLE_INPUT_ROOT.exists():
        patterns = [
            f'benchmarks/{DATASET}/{DATASET_JSON_DIR}',
            f'*/benchmarks/{DATASET}/{DATASET_JSON_DIR}',
            f'*/{DATASET}/{DATASET_JSON_DIR}',
            DATASET_JSON_DIR,
            f'*/{DATASET_JSON_DIR}',
            f'*/{DATASET}',
            f'**/{DATASET_JSON_DIR}',
        ]
        for pattern in patterns:
            candidates.extend(sorted(KAGGLE_INPUT_ROOT.glob(pattern)))

    for candidate in candidates:
        possible_paths = [candidate]
        if candidate.is_dir() and candidate.name != DATASET_JSON_DIR:
            possible_paths.append(candidate / DATASET_JSON_DIR)

        for possible in possible_paths:
            possible = possible.resolve()
            if possible in seen:
                continue
            seen.add(possible)
            checked.append(str(possible))
            if _count_json_files(possible) > 0:
                return str(possible)

    raise RuntimeError(
        "[FATAL] Dataset not found. Add a Kaggle Input dataset containing "
        f"benchmarks/{DATASET}/{DATASET_JSON_DIR}/*.json or a {DATASET_JSON_DIR} folder. Checked:\n  "
        + "\n  ".join(checked)
    )


DATASET_PATH = _resolve_dataset_path()

# Problem list for this group (from repo)
PROBLEM_LIST = f"{REPO_DIR}/kaggle_setup/nosbench_splits/nosbench_group{GROUP}_problems.txt"

print(f"[OK] Dataset found: {DATASET_PATH}")
print(f"[OK] Results will save to: {OUTPUT_DIR}")


## 2. Prepare The Repository

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

repo_path = Path(REPO_DIR)
REPO_COMMIT = "PUT_RELEASE_COMMIT_SHA_HERE"
if repo_path.exists():
    shutil.rmtree(repo_path)

print(f"Cloning: {REPO_GIT_URL} @ {REPO_COMMIT}")
subprocess.run(["git", "clone", "--depth", "1", REPO_GIT_URL, str(repo_path)], check=True)
subprocess.run(["git", "checkout", REPO_COMMIT], check=True, cwd=str(repo_path))
sys.path.insert(0, str(repo_path))
print(f"[OK] Repository ready")

## 3. Install Dependencies

In [ ]:
%%bash
cd /kaggle/working/mpecss-kaggle
pip install -q --upgrade pip
pip install -q -e .
echo "[OK] Installation complete"

## 4. Data Path Setup

In [ ]:
import os

DATASET_PATH = _resolve_dataset_path()
problem_count = _count_json_files(DATASET_PATH)

print(f"Dataset path: {DATASET_PATH}")
print(f"Problem list: {PROBLEM_LIST}")

if problem_count == 0:
    raise RuntimeError(f"[FATAL] Dataset not found: {DATASET_PATH}")
if problem_count != EXPECTED_PROBLEM_COUNT:
    print(f"[WARN] Expected {EXPECTED_PROBLEM_COUNT} total problems in dataset, found {problem_count}")
print(f"[OK] Total problems in dataset: {problem_count}")

if os.path.exists(PROBLEM_LIST):
    with open(PROBLEM_LIST) as f:
        group_count = sum(1 for line in f if line.strip() and not line.startswith('#'))
    print(f"[OK] Problems in Group {GROUP}: {group_count}")
else:
    print(f"[ERROR] Problem list not found!")


## 5. Inspect The Kaggle Instance

In [ ]:
import multiprocessing
import platform
import psutil

print("=" * 60)
print("System Information")
print("=" * 60)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"Logical CPU cores: {multiprocessing.cpu_count()}")

mem = psutil.virtual_memory()
print(f"Total RAM: {mem.total / 1024**3:.1f} GB")
print(f"Available RAM: {mem.available / 1024**3:.1f} GB")
print("=" * 60)

## 6. Run Preflight Checks

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/mpecss-kaggle
python mpecss/helpers/preflight_checks.py

## 7. Load The Runner Helper

In [ ]:
import subprocess
import sys


def run_benchmark(resume=False, summary=False, retry_failed=False):
    cmd = [
        sys.executable,
        f"{str(REPO_DIR)}/kaggle_setup/resumable_benchmark.py",
        "--dataset", DATASET,
        "--repo-dir", str(REPO_DIR),
        "--tag", RUN_TAG,
        "--workers", str(WORKERS),
        "--timeout", str(TIMEOUT),
        "--path", str(DATASET_PATH),
        "--problem-list", str(PROBLEM_LIST),
        "--output-dir", str(OUTPUT_DIR),  # CRITICAL: Save to persistent location
    ]
    if NUM_PROBLEM is not None:
        cmd.extend(["--num-problems", str(NUM_PROBLEM)])
    if SAVE_LOGS:
        cmd.append("--save-logs")
    if SHUFFLE:
        cmd.append("--shuffle")
    if resume or retry_failed:
        cmd.append("--resume-latest")
    if retry_failed:
        cmd.append("--retry-failed")
    if summary:
        cmd.append("--summary-only")

    print("+ " + " ".join(str(x) for x in cmd))
    subprocess.run(cmd)


## 8. Launch A Fresh Run

In [ ]:
run_benchmark()

## 9. Resume (Manual, After Restart)


In [ ]:
# run_benchmark(resume=True)


## 10. Retry Only Failed Problems (Manual)


In [ ]:
# run_benchmark(resume=True, retry_failed=True)


## 11. Progress Summary


In [ ]:
run_benchmark(summary=True)

## 12. Copy Results


In [ ]:
from pathlib import Path
import shutil
import os

output_dir = Path(OUTPUT_DIR)
if not output_dir.exists():
    print(f"[WARN] Output directory not found: {output_dir}")
else:
    zip_name = f"{globals().get('RUN_TAG', globals().get('RUN_TAG_PREFIX', 'results'))}_results"
    temp_dir = output_dir.parent / zip_name
    os.rename(output_dir, temp_dir)
    archive = shutil.make_archive(str(temp_dir), "zip", root_dir=str(temp_dir.parent), base_dir=zip_name)
    os.rename(temp_dir, output_dir)
    print(f"[OK] Results directory: {output_dir}")
    print(f"[OK] Download archive: {archive}")
    for item in sorted(output_dir.iterdir()):
        print(item)


## 13. Software Versions


In [ ]:
import casadi
import matplotlib
import numpy
import pandas
import platform
import psutil
import scipy

print("Software Versions")
print("=" * 40)
print(f"Python: {platform.python_version()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"CasADi: {casadi.__version__}")
print(f"psutil: {psutil.__version__}")
print("=" * 40)